In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
import numpy as np

In [ ]:
N = 100
P = 100

stats_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/summary_stats/')

results_path = os.path.expanduser(f'~/hcda_pyspoc_project/py-spoc/working_analyses/hjResearch_2025/hpc_pyspoc_scripts/results/concat/')
dataset_folder = f"gtex_N{N}_P{P}"
dm_file = 'datamatrix__3configs__no_filter.parquet'

data_path = os.path.join(results_path,dataset_folder,dm_file)

In [ ]:
data_matrix = pd.read_parquet(data_path)

In [ ]:
data_matrix_names = data_matrix.columns.str.replace(stats_path, ".", regex=False)
data_matrix.columns = data_matrix_names

In [ ]:
# rem duplicate cols
dup = data_matrix.T.duplicated()
df_dedup = data_matrix.loc[:, ~dup]

# rem constant cols
df_dedup = df_dedup.loc[:,~(df_dedup.nunique() <= 1)]

df_dedup_scaled = (df_dedup - df_dedup.mean())/df_dedup.std()
print(f"dropped {dup.sum()} duplicate columns")

In [ ]:
# dup[dup]

In [ ]:
data_matrix.shape, df_dedup.shape

In [ ]:
pyspoc_stat_fam = pd.Series(data_matrix.columns.str.split(".").str[-6:-1].str.join("."), index=data_matrix.columns)

In [ ]:
# pyspoc_stat_fam

In [ ]:
df_module_cluster = df_dedup

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


In [ ]:
X = StandardScaler().fit_transform(data_matrix)

n_pcs = 20
pca = PCA(n_components=n_pcs)
Z = pca.fit_transform(X)

groups = df_module_cluster.index.str.split('_N').str[0].str.replace('gtex_brain_','')
grp_keys = {v: brain_groups[v] for v in groups.unique()}
coarse = groups.map(grp_keys)

In [ ]:
brain_groups = {
    "cerebellum": "cerebellar", "cerebellar_hemisphere": "cerebellar",
    "cortex": "cortical", "frontal_cortex_ba9": "cortical", "anterior_cingulate_cortex_ba24": "cortical",
    "caudate_basal_ganglia": "striatal", "putamen_basal_ganglia": "striatal",
    "nucleus_accumbens_basal_ganglia": "striatal",
    "hippocampus": "limbic", "amygdala": "limbic", "hypothalamus": "limbic",
    "spinal_cord_cervical_c_1": "glia_rich", "substantia_nigra": "glia_rich",
}

In [ ]:
# coarse.unique()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for g in coarse.unique():
    m = coarse == g
    ax.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.7, label=g)

ev = pca.explained_variance_ratio_
ax.set_xlabel(f"PC1 ({ev[0]:.0%})"); ax.set_ylabel(f"PC2 ({ev[1]:.0%})")
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()

In [ ]:
y, names = pd.factorize(coarse)
acc = cross_val_score(LogisticRegression(max_iter=5000), X = X, y = y, cv=10)
print(f"accuracy {acc.mean():.2f} +/- {acc.std():.2f}, chance {groups.value_counts(normalize=True).max():.2f}")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict

df_brain = data_matrix                                     # your brain datasets x statistics

pipe = make_pipeline(StandardScaler(),
                     LogisticRegression(max_iter=5000))

tissue = pd.Series(df_brain.index.str.split('_N').str[0].str.replace('gtex_brain_',''))

y_coarse = tissue.map(brain_groups)
assert y_coarse.notna().all(), tissue[y_coarse.isna()].unique()   # every tissue mapped?

pred = cross_val_predict(pipe, np.asarray(df_brain), y_coarse,
                         groups=tissue, cv=LeaveOneGroupOut())

print(f"accuracy {(pred == y_coarse).mean():.2f}")
print(pd.crosstab(tissue, pred, rownames=["held-out tissue"], colnames=["predicted group"]))

In [ ]:
from sklearn.manifold import TSNE

E = TSNE(n_components=2, init="pca").fit_transform(Z)

fig, ax = plt.subplots(figsize=(6, 3))
for g in sorted(groups.unique()):
    m = (groups == g)
    ax.scatter(E[m, 0], E[m, 1], s=10, alpha=0.7, label=g)
ax.set_xticks([]); ax.set_yticks([]); ax.set_title("t-SNE")
ax.legend(fontsize=8, markerscale=1.5, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout(); plt.show()